In [ ]:
!pip install -q segmentation-models-pytorch==0.5.0 pycocotools 2>&1 | tail -1

In [ ]:
import os
for d in ['solarfil','solarfil5']: os.makedirs(d,exist_ok=True)

In [ ]:
%%writefile solarfil/__init__.py
"""solarfil: instance segmentation of solar filaments in GONG H-alpha images.

Pipeline
--------
1. preprocess  : limb-darkening flattening + contrast channels (preprocess.py)
2. targets     : multi-annotator soft foreground + instance-border maps (annotations.py)
3. model       : U-Net (segmentation_models_pytorch) with 2 outputs: foreground, border (model.py)
4. training    : high-resolution crops, Dice+BCE, EMA, GroupKFold by photo (train.py)
5. inference   : full-disk prediction, dihedral TTA, fold ensemble (inference.py)
6. instances   : threshold -> fragment linking -> border splitting -> hole filling (postprocess.py)
7. evaluation  : exact re-implementation of the official Panoptic Quality metric (metric.py)
"""
__version__ = "2.0.0"

In [ ]:
%%writefile solarfil/annotations.py
"""MAGFiLO COCO annotations -> ground-truth RLEs and soft training targets.

Important dataset facts (verified on the 2026 Kaggle train split):
  * 707 unique photos, 1154 COCO `images`: a photo can be annotated by 1-3 annotators.
    Each annotator copy has its own image id "<annotator>-<photo stem>".
  * The official metric scores the prediction for a photo against EVERY annotator copy
    separately (see metric.py), so the best prediction is a consensus of annotators.
  * Every annotation is a single filled polygon.
"""
from __future__ import annotations

import json
import os
from collections import defaultdict
from dataclasses import dataclass, field

import cv2
import numpy as np
from pycocotools import mask as mask_util

NATIVE = 2048


@dataclass
class Photo:
    stem: str
    file_name: str
    # annotator image id -> list of polygons (flat [x0, y0, x1, y1, ...])
    annotators: dict = field(default_factory=dict)


def load_photos(json_path: str) -> dict[str, Photo]:
    with open(json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)
    anns = defaultdict(list)
    for a in coco["annotations"]:
        anns[a["image_id"]].append(a)
    photos: dict[str, Photo] = {}
    for im in coco["images"]:
        stem = os.path.splitext(im["file_name"])[0]
        p = photos.setdefault(stem, Photo(stem=stem, file_name=im["file_name"]))
        polys = []
        for a in anns.get(im["id"], []):
            seg = a["segmentation"]
            if isinstance(seg, list):
                polys.extend([s for s in seg if len(s) >= 6])
        p.annotators[im["id"]] = polys
    return photos


def poly_to_rle(poly: list, h: int = NATIVE, w: int = NATIVE) -> dict:
    rles = mask_util.frPyObjects([poly], h, w)
    return mask_util.merge(rles)


def photo_gt_rles(photo: Photo) -> dict[str, list[dict]]:
    """annotator id -> list of COCO RLE dicts (one per filament), exactly like annToMask."""
    return {aid: [poly_to_rle(p) for p in polys] for aid, polys in photo.annotators.items()}


def soft_targets(photo: Photo, size: int, border_px: int = 5) -> tuple[np.ndarray, np.ndarray]:
    """Soft foreground and instance-border targets at (size, size), averaged over annotators.

    foreground: fraction of annotators (and of the downsampled pixel area) covered by a filament.
    border    : pixels where dilated masks of two different filaments of the same annotator meet.
                Teaches the network where touching filaments must be split.
    """
    fg = np.zeros((size, size), np.float32)
    bd = np.zeros((size, size), np.float32)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * border_px + 1, 2 * border_px + 1))
    n_ann = max(len(photo.annotators), 1)
    for polys in photo.annotators.values():
        union = np.zeros((NATIVE, NATIVE), np.uint8)
        count = np.zeros((NATIVE, NATIVE), np.uint8)
        for p in polys:
            m = mask_util.decode(poly_to_rle(p))
            union |= m
            if len(polys) > 1:
                count += cv2.dilate(m, k)
        fg += cv2.resize(union.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
        if len(polys) > 1:
            b = (count >= 2).astype(np.float32)
            bd += cv2.resize(b, (size, size), interpolation=cv2.INTER_AREA)
    return fg / n_ann, np.clip(bd / n_ann, 0, 1)


def gt_frame(photos: dict[str, Photo], stems: list[str] | None = None):
    """Ground truth in the official submission format (annotator-prefixed filament ids)."""
    import pandas as pd

    rows = []
    for stem in stems or list(photos):
        for aid, polys in photos[stem].annotators.items():
            for i, p in enumerate(polys, 1):
                rle = poly_to_rle(p)
                rows.append({"filament_id": f"{aid}_{i}", "segmentation_rle": rle["counts"].decode()})
    return pd.DataFrame(rows)

In [ ]:
%%writefile solarfil/data.py
"""Caching, folds and PyTorch datasets."""
from __future__ import annotations

import os
import random
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from .annotations import Photo, soft_targets
from .preprocess import load_input


# ----------------------------------------------------------------------------- folds
def make_folds(stems: list[str], n_folds: int = 5, seed: int = 42) -> pd.DataFrame:
    """Group by observation date: same-day photos from different GONG sites see the same filaments."""
    df = pd.DataFrame({"stem": sorted(stems)})
    df["group"] = df["stem"].str[:8]
    groups = df["group"].unique().tolist()
    rng = random.Random(seed)
    rng.shuffle(groups)
    gfold = {g: i % n_folds for i, g in enumerate(groups)}
    df["fold"] = df["group"].map(gfold)
    return df


# ----------------------------------------------------------------------------- cache
def _cache_one(args):
    photo, img_path, cache_dir, size, border_px = args
    xp = os.path.join(cache_dir, f"{photo.stem}_x.npy")
    yp = os.path.join(cache_dir, f"{photo.stem}_y.npy")
    if os.path.exists(xp) and os.path.exists(yp):
        return photo.stem
    x = load_input(img_path, size)
    fg, bd = soft_targets(photo, size, border_px=border_px)
    y = np.dstack([np.round(fg * 255), np.round(bd * 255)]).astype(np.uint8)
    np.save(xp, x)
    np.save(yp, y)
    return photo.stem


def build_cache(photos: dict[str, Photo], img_dir: str, cache_dir: str, size: int,
                border_px: int = 5, workers: int = 4):
    os.makedirs(cache_dir, exist_ok=True)
    jobs = [(p, os.path.join(img_dir, p.file_name), cache_dir, size, border_px) for p in photos.values()]
    if workers <= 1:
        for j in jobs:
            _cache_one(j)
    else:
        with ProcessPoolExecutor(workers) as ex:
            for i, _ in enumerate(ex.map(_cache_one, jobs, chunksize=4)):
                if i % 100 == 0:
                    print(f"cache {i}/{len(jobs)}", flush=True)


def build_test_cache(paths: list[str], cache_dir: str, size: int, workers: int = 4):
    os.makedirs(cache_dir, exist_ok=True)

    todo = [p for p in paths
            if not os.path.exists(os.path.join(cache_dir, os.path.splitext(os.path.basename(p))[0] + "_x.npy"))]
    with ProcessPoolExecutor(max(workers, 1)) as ex:
        for p, x in zip(todo, ex.map(load_input, todo, [size] * len(todo))):
            np.save(os.path.join(cache_dir, os.path.splitext(os.path.basename(p))[0] + "_x.npy"), x)


# ----------------------------------------------------------------------------- augmentation
def dihedral(a: np.ndarray, k: int) -> np.ndarray:
    """k in 0..7: rot90 by k%4, then horizontal flip if k>=4."""
    a = np.rot90(a, k % 4, axes=(0, 1))
    if k >= 4:
        a = a[:, ::-1]
    return a


def photometric(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    x = x.astype(np.float32) / 255.0
    if rng.random() < 0.8:  # contrast / brightness / gamma, per channel jitter
        c = rng.uniform(0.75, 1.3)
        b = rng.uniform(-0.08, 0.08)
        g = rng.uniform(0.75, 1.35)
        x = np.clip((x - 0.5) * c + 0.5 + b, 0, 1) ** g
    if rng.random() < 0.3:
        x = x + rng.normal(0, rng.uniform(0.005, 0.03), x.shape).astype(np.float32)
    if rng.random() < 0.2:
        import cv2
        x = cv2.GaussianBlur(x, (0, 0), rng.uniform(0.5, 1.2))
    return np.clip(x, 0, 1)


# ----------------------------------------------------------------------------- datasets
class CropDataset(Dataset):
    """Random crops from cached full-disk arrays; `pos_frac` of crops are centred on a filament."""

    def __init__(self, stems, cache_dir, crop=768, samples_per_epoch=None, pos_frac=0.7, seed=0):
        self.stems = list(stems)
        self.cache_dir = cache_dir
        self.crop = crop
        self.n = samples_per_epoch or 2 * len(self.stems)
        self.pos_frac = pos_frac
        self.seed = seed
        self._fgidx = {}

    def __len__(self):
        return self.n

    def _arrays(self, stem):
        x = np.load(os.path.join(self.cache_dir, f"{stem}_x.npy"), mmap_mode="r")
        y = np.load(os.path.join(self.cache_dir, f"{stem}_y.npy"), mmap_mode="r")
        return x, y

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed * 1_000_003 + i + random.getrandbits(31)) % 2**32)
        stem = self.stems[rng.integers(len(self.stems))]
        x, y = self._arrays(stem)
        S, c = x.shape[0], self.crop
        if rng.random() < self.pos_frac:
            if stem not in self._fgidx:
                ys, xs = np.nonzero(np.asarray(y[::8, ::8, 0]) > 64)
                self._fgidx[stem] = (ys * 8, xs * 8)
            ys, xs = self._fgidx[stem]
            if len(ys):
                j = rng.integers(len(ys))
                cy = int(ys[j] + rng.integers(-c // 3, c // 3 + 1))
                cx = int(xs[j] + rng.integers(-c // 3, c // 3 + 1))
            else:
                cy, cx = rng.integers(c // 2, S - c // 2 + 1, size=2)
        else:
            cy, cx = rng.integers(c // 2, S - c // 2 + 1, size=2)
        y0 = int(np.clip(cy - c // 2, 0, S - c))
        x0 = int(np.clip(cx - c // 2, 0, S - c))
        xc = np.asarray(x[y0:y0 + c, x0:x0 + c])
        yc = np.asarray(y[y0:y0 + c, x0:x0 + c]).astype(np.float32) / 255.0
        k = int(rng.integers(8))
        xc, yc = dihedral(xc, k), dihedral(yc, k)
        xc = photometric(np.ascontiguousarray(xc), rng)
        return (torch.from_numpy(np.ascontiguousarray(xc.transpose(2, 0, 1))),
                torch.from_numpy(np.ascontiguousarray(yc.transpose(2, 0, 1))))


def load_full(stem: str, cache_dir: str) -> np.ndarray:
    """Full cached input as float32 CHW tensor in [0,1]."""
    x = np.load(os.path.join(cache_dir, f"{stem}_x.npy"))
    return torch.from_numpy(x.transpose(2, 0, 1).astype(np.float32) / 255.0)

In [ ]:
%%writefile solarfil/infer.py
"""Predict the test set and write submission.csv.

python -m solarfil.infer --data_root <root> --ckpts work/*.pth --postcfg postcfg.json --out submission.csv
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import time

import numpy as np
import pandas as pd
import torch
from pycocotools import mask as mask_util

from .data import build_test_cache, load_full
from .model import FilamentNet, predict_full
from .postprocess import PostCfg, instances_rle


def load_models(paths, dev):
    groups = {}
    for p in paths:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        a = ck["args"]
        m = FilamentNet(a["arch"], a["encoder"], weights=None)
        m.load_state_dict(ck["state_dict"])
        groups.setdefault(a["size"], []).append(m.to(dev).eval())
        print(f"loaded {os.path.basename(p)} (val PQ {ck.get('pq', float('nan')):.4f}, size {a['size']})")
    return groups


def check_submission(df: pd.DataFrame):
    assert list(df.columns) == ["filament_id", "segmentation_rle"]
    assert df["filament_id"].is_unique
    assert not df.isnull().any().any()
    for stem, g in df.groupby(df["filament_id"].str.split("_", n=1).str[0]):
        rles = [{"size": [2048, 2048], "counts": r.encode()} for r in g["segmentation_rle"]]
        if len(rles) > 1:
            inter = mask_util.area(mask_util.merge(rles, intersect=False))
            assert inter == mask_util.area(rles).sum(), f"overlapping masks in {stem}"


def main(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True)
    a.add_argument("--ckpts", nargs="+", required=True)
    a.add_argument("--postcfg", default="")
    a.add_argument("--work", default="work")
    a.add_argument("--tta", type=int, default=8)
    a.add_argument("--out", default="submission.csv")
    a.add_argument("--save_probs", default="")
    a.add_argument("--rescorer", default="", help="pickle from rescore.py (overrides min_score/min_area)")
    args = a.parse_args(argv)

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = PostCfg(**json.load(open(args.postcfg))["cfg"]) if args.postcfg else PostCfg()
    resc = None
    if args.rescorer:
        import pickle
        resc = pickle.load(open(args.rescorer, "rb"))
        cfg = PostCfg(**resc["cand_cfg"])
        print(f"rescorer thr={resc['thr']:.3f} (CV PQ {resc['cv_pq']:.4f})")
    print("post-processing:", cfg)
    paths = sorted(glob.glob(os.path.join(args.data_root, "test", "test_images", "*.jp*g")))
    groups = load_models(args.ckpts, dev)
    for size in groups:
        build_test_cache(paths, os.path.join(args.work, f"test_cache_{size}"), size)

    rows, t0 = [], time.time()
    for path in paths:
        stem = os.path.splitext(os.path.basename(path))[0]
        probs, top = [], max(groups)
        for size, models in groups.items():
            p = predict_full(models, load_full(stem, os.path.join(args.work, f"test_cache_{size}")), tta=args.tta)
            if size != top:  # models trained at different resolutions: average at the largest one
                p = torch.nn.functional.interpolate(p[None], size=(top, top), mode="bilinear", align_corners=False)[0]
            probs.append((p.numpy(), len(models)))
        p = sum(q * n for q, n in probs) / sum(n for _, n in probs)
        if args.save_probs:
            os.makedirs(args.save_probs, exist_ok=True)
            np.savez_compressed(os.path.join(args.save_probs, f"{stem}.npz"), p=np.round(p * 255).astype(np.uint8))
        for i, r in enumerate(instances_rle(p[0], p[1], cfg, rescorer=resc), 1):
            rows.append({"filament_id": f"{stem}_{i}", "segmentation_rle": r})
    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    check_submission(df)
    df.to_csv(args.out, index=False)
    n_img = df["filament_id"].str.split("_", n=1).str[0].nunique()
    print(f"wrote {args.out}: {len(df)} filaments in {n_img}/{len(paths)} images ({time.time() - t0:.0f}s)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile solarfil/metric.py
"""Exact re-implementation of the official Panoptic Quality metric (self-evaluation notebook).

Official behaviour reproduced here:
  * GT rows are keyed "<annotator>-<stem>_<k>", prediction rows "<stem>_<k>".
  * For every annotator copy of a photo, ALL predictions of that photo are compared with
    that annotator's filaments; TP iff IoU > 0.5 (strict), unmatched preds = FP,
    unmatched GT = FN.  PQ = sum(IoU_TP) / (TP + FP/2 + FN/2), pooled over everything.
  * Photos without GT (e.g. the other half of the test set) are ignored.
IoUs are computed on RLEs with pycocotools (identical numbers, ~100x faster than dense
2048x2048 matmuls).
"""
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass

import numpy as np
from pycocotools import mask as mask_util

SIZE = [2048, 2048]


def _as_rle(r):
    if isinstance(r, dict):
        return r
    return {"size": SIZE, "counts": r.encode() if isinstance(r, str) else r}


@dataclass
class PQStats:
    iou_sum: float = 0.0
    tp: int = 0
    fp: int = 0
    fn: int = 0

    def add(self, o: "PQStats"):
        self.iou_sum += o.iou_sum
        self.tp += o.tp
        self.fp += o.fp
        self.fn += o.fn
        return self

    @property
    def pq(self) -> float:
        d = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.iou_sum / d if d > 0 else 0.0

    @property
    def sq(self) -> float:
        return self.iou_sum / self.tp if self.tp else 0.0

    @property
    def dq(self) -> float:
        d = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.tp / d if d > 0 else 0.0

    def __repr__(self):
        return (f"PQ={self.pq:.4f} (SQ={self.sq:.4f} DQ={self.dq:.4f}) "
                f"TP={self.tp} FP={self.fp} FN={self.fn}")


def image_stats(gt_rles: list, pred_rles: list) -> PQStats:
    """Stats for one annotator copy vs. the predictions of that photo."""
    n_gt, n_pred = len(gt_rles), len(pred_rles)
    if n_gt == 0:
        return PQStats(fp=n_pred)
    if n_pred == 0:
        return PQStats(fn=n_gt)
    g = [_as_rle(r) for r in gt_rles]
    p = [_as_rle(r) for r in pred_rles]
    iou = np.asarray(mask_util.iou(p, g, [0] * n_gt))  # (n_pred, n_gt)
    hit = iou > 0.5
    return PQStats(
        iou_sum=float(iou[hit].sum()),
        tp=int(hit.sum()),
        fp=int((hit.sum(axis=1) == 0).sum()),
        fn=int((hit.sum(axis=0) == 0).sum()),
    )


def photo_stats(gt_by_annotator: dict, pred_rles: list) -> PQStats:
    s = PQStats()
    for rles in gt_by_annotator.values():
        if len(rles):
            s.add(image_stats(rles, pred_rles))
    return s


def score_frames(gt_df, pred_df) -> PQStats:
    """Score two DataFrames in the official submission format."""
    gt = defaultdict(lambda: defaultdict(list))
    for fid, rle in zip(gt_df["filament_id"], gt_df["segmentation_rle"]):
        ann_img = fid.split("_", 1)[0]
        stem = ann_img.split("-", 1)[1]
        gt[stem][ann_img].append(rle)
    pred = defaultdict(list)
    for fid, rle in zip(pred_df["filament_id"], pred_df["segmentation_rle"]):
        pred[fid.split("_", 1)[0]].append(rle)
    total = PQStats()
    for stem, anns in gt.items():
        total.add(photo_stats(anns, pred.get(stem, [])))
    return total

In [ ]:
%%writefile solarfil/model.py
"""Network, losses and (TTA) inference."""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class FilamentNet(nn.Module):
    """U-Net with 2 output channels: 0 = filament foreground, 1 = instance border."""

    def __init__(self, arch="Unet", encoder="efficientnet-b4", weights="imagenet"):
        super().__init__()
        kw = dict(encoder_name=encoder, encoder_weights=weights, in_channels=3, classes=2)
        if arch in ("Unet", "UnetPlusPlus"):
            kw["decoder_attention_type"] = "scse"
        self.net = getattr(smp, arch)(**kw)
        self.register_buffer("mean", MEAN.clone(), persistent=False)
        self.register_buffer("std", STD.clone(), persistent=False)

    def forward(self, x):  # x in [0,1]
        return self.net((x - self.mean) / self.std)


def soft_dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2))
    den = p.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    return (1 - (2 * inter + eps) / (den + eps)).mean()


class FilamentLoss(nn.Module):
    def __init__(self, border_weight=0.5, pos_weight=3.0):
        super().__init__()
        self.bw = border_weight
        self.pw = pos_weight

    def forward(self, logits, target):
        fg_l, bd_l = logits[:, 0], logits[:, 1]
        fg_t, bd_t = target[:, 0], target[:, 1]
        pw = torch.tensor(self.pw, device=logits.device)
        loss = F.binary_cross_entropy_with_logits(fg_l, fg_t, pos_weight=pw) + soft_dice_loss(fg_l, fg_t)
        if self.bw > 0:
            loss = loss + self.bw * (F.binary_cross_entropy_with_logits(bd_l, bd_t, pos_weight=pw * 3)
                                     + 0.5 * soft_dice_loss(bd_l, bd_t))
        return loss


@torch.no_grad()
def predict_full(models, x: torch.Tensor, tta: int = 8, amp: bool = True) -> torch.Tensor:
    """x: (3,H,W) in [0,1]. Returns averaged sigmoid probs (2,H,W) on CPU.

    tta: number of dihedral transforms (1, 2, 4 or 8)."""
    if not isinstance(models, (list, tuple)):
        models = [models]
    dev = next(models[0].parameters()).device
    xb = x.unsqueeze(0).to(dev)
    acc = None
    n = 0
    for k in range(tta):
        r, f = k % 4, k >= 4
        xt = torch.rot90(xb, r, dims=(2, 3))
        if f:
            xt = torch.flip(xt, dims=(3,))
        for m in models:
            with torch.autocast(device_type=dev.type, enabled=amp and dev.type == "cuda"):
                p = torch.sigmoid(m(xt).float())
            if f:
                p = torch.flip(p, dims=(3,))
            p = torch.rot90(p, -r, dims=(2, 3))
            acc = p if acc is None else acc + p
            n += 1
    return (acc / n)[0].float().cpu()


class EMA:
    def __init__(self, model, decay=0.999):
        import copy
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)
            else:
                v.copy_(msd[k])

In [ ]:
%%writefile solarfil/postprocess.py
"""Probability maps -> non-overlapping filament instances (COCO RLE at 2048x2048).

Panoptic Quality punishes two failure modes that plain "threshold + connected components"
produces on filaments:
  * fragmentation: one long faint filament breaks into several blobs -> every blob is an FP
    and the real filament an FN.            -> fix: link fragments closer than `link_px`.
  * over-merging : two neighbouring filaments touch -> one blob, IoU < 0.5 with both.
                                             -> fix: cut along the predicted border map.
Annotators draw filled polygons, so instance holes are filled.
All parameters are tuned on out-of-fold predictions with the official metric (tune.py).
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

import cv2
import numpy as np
from pycocotools import mask as mask_util
from scipy import ndimage as ndi

NATIVE = 2048


@dataclass
class PostCfg:
    thr: float = 0.45          # foreground probability threshold
    close_px: int = 0          # morphological closing radius before labelling
    min_cc: int = 40           # drop tiny specks before linking (native px)
    link_px: int = 8           # fragments closer than this are one filament (native px)
    border_thr: float = 1.01   # >1 disables border splitting
    min_area: int = 250        # final instance area filter (native px)
    min_score: float = 0.50    # mean foreground prob inside instance
    seed_thr: float = 1.01     # <=1: split instances by watershed from high-confidence seeds
    fill_holes: bool = True

    def as_dict(self):
        return asdict(self)


def _disk(r: int):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def upsample(p: np.ndarray, size: int = NATIVE) -> np.ndarray:
    p = np.ascontiguousarray(p, dtype=np.float32)
    if p.shape[0] == size:
        return p
    return cv2.resize(p, (size, size), interpolation=cv2.INTER_LINEAR)


def instances(fg: np.ndarray, border: np.ndarray | None, cfg: PostCfg, want_feats: bool = False):
    """fg/border: float maps in [0,1] at any square resolution. Returns list of uint8 masks (2048²)."""
    fg = upsample(fg)
    binary = (fg > cfg.thr).astype(np.uint8)
    if cfg.close_px > 0:
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, _disk(cfg.close_px))
    if not binary.any():
        return ([], []) if want_feats else []

    # 1) remove specks
    n, lab, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= cfg.min_cc
    binary = keep[lab].astype(np.uint8)
    if not binary.any():
        return ([], []) if want_feats else []

    # 2) link nearby fragments, 3) cut along predicted borders
    core = cv2.dilate(binary, _disk(cfg.link_px // 2)) if cfg.link_px > 1 else binary.copy()
    use_border = border is not None and cfg.border_thr <= 1.0
    if use_border:
        cut = upsample(border) > cfg.border_thr
        core[cut] = 0
    if cfg.seed_thr <= 1.0:
        # seeds = linked high-confidence cores; grow them over the foreground along the probability
        seeds = ((fg > cfg.seed_thr) & (core > 0)).astype(np.uint8)
        if cfg.link_px > 1:
            seeds = cv2.dilate(seeds, _disk(cfg.link_px // 2)) & core
        n, markers = cv2.connectedComponents(seeds, connectivity=8)
        from skimage.segmentation import watershed
        lab = watershed(-fg, markers, mask=core > 0).astype(np.int32)
        # components without any seed keep their own label
        n2, lab2 = cv2.connectedComponents(((core > 0) & (lab == 0)).astype(np.uint8), connectivity=8)
        lab[lab2 > 0] = lab2[lab2 > 0] + n
    else:
        n, lab = cv2.connectedComponents(core, connectivity=8)
        lab = lab.astype(np.int32)
    lab[binary == 0] = 0
    if use_border:
        # border pixels that belong to the foreground: give them to the nearest instance
        orphan = (binary > 0) & (lab == 0)
        if orphan.any() and n > 1:
            _, (iy, ix) = ndi.distance_transform_edt(lab == 0, return_indices=True)
            lab[orphan] = lab[iy[orphan], ix[orphan]]

    # 4) per-instance filtering
    out, feats = [], []
    objs = ndi.find_objects(lab)
    n_obj = sum(o is not None for o in objs)
    bd_full = upsample(border) if (border is not None and want_feats) else None
    for i, sl in enumerate(objs, 1):
        if sl is None:
            continue
        m = lab[sl] == i
        if cfg.fill_holes:
            m = ndi.binary_fill_holes(m)
            m &= (lab[sl] == i) | (lab[sl] == 0)  # never steal pixels of another instance
        area = int(m.sum())
        if area < cfg.min_area:
            continue
        pv = fg[sl][m]
        if float(pv.mean()) < cfg.min_score:
            continue
        full = np.zeros((NATIVE, NATIVE), np.uint8)
        full[sl][m] = 1
        out.append(full)
        if want_feats:
            feats.append(_features(m, sl, pv, bd_full[sl][m] if bd_full is not None else None, n_obj))
    return (out, feats) if want_feats else out


FEATURES = ["log_area", "mean_p", "max_p", "p10", "p50", "p90", "std_p", "f50", "f70", "f90",
            "mean_bd", "h", "w", "fill", "elong", "r_center", "n_obj"]


def _features(m, sl, pv, bv, n_obj):
    h = sl[0].stop - sl[0].start
    w = sl[1].stop - sl[1].start
    ys, xs = np.nonzero(m)
    cy, cx = ys.mean() + sl[0].start, xs.mean() + sl[1].start
    if len(ys) > 2:
        ev = np.linalg.eigvalsh(np.cov(np.vstack([ys, xs])))
        elong = float(np.sqrt(max(ev[-1], 1e-6) / max(ev[0], 1e-6)))
    else:
        elong = 1.0
    q = np.percentile(pv, [10, 50, 90])
    return [np.log(m.sum()), float(pv.mean()), float(pv.max()), q[0], q[1], q[2], float(pv.std()),
            float((pv > .5).mean()), float((pv > .7).mean()), float((pv > .9).mean()),
            float(bv.mean()) if bv is not None else 0.0, h, w, m.sum() / max(h * w, 1), min(elong, 50.0),
            float(np.hypot(cy - 1024, cx - 1024) / 1024), n_obj]


def load_avg(paths: list[str]) -> np.ndarray:
    """Average uint8 probability npz files (2,S,S) of possibly different resolutions."""
    ps = [np.load(f)["p"].astype(np.float32) / 255.0 for f in paths]
    top = max(p.shape[-1] for p in ps)
    ps = [p if p.shape[-1] == top else np.stack([upsample(c, top) for c in p]) for p in ps]
    return np.mean(ps, axis=0)


def encode(masks: list[np.ndarray]) -> list[str]:
    return [mask_util.encode(np.asfortranarray(m))["counts"].decode() for m in masks]


def instances_rle(fg, border, cfg: PostCfg, rescorer=None) -> list[str]:
    """rescorer: optional dict {"model": fitted regressor, "thr": float} (see rescore.py)."""
    if rescorer is None:
        return encode(instances(fg, border, cfg))
    masks, feats = instances(fg, border, cfg, want_feats=True)
    if not masks:
        return []
    s = rescorer["model"].predict(np.asarray(feats, np.float32))
    return encode([m for m, v in zip(masks, s) if v >= rescorer["thr"]])

In [ ]:
%%writefile solarfil/preprocess.py
"""Image preprocessing for GONG H-alpha full-disk observations.

Filaments are dark, elongated absorption features on the solar disk. Their contrast
is low and the disk brightness falls off towards the limb (limb darkening) and varies
between the six GONG sites. We therefore build a 3-channel input:

  ch0  global intensity (percentile-normalised)
  ch1  flattened image  = image / smooth background  (removes limb darkening / gradients)
  ch2  local contrast    = CLAHE of the flattened image

All channels are uint8 so they can be cached cheaply.
"""
from __future__ import annotations

import cv2
import numpy as np

NATIVE = 2048


def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img


def disk_mask(img: np.ndarray) -> np.ndarray:
    """Binary mask of the solar disk (Otsu on a blurred image, largest component, filled)."""
    small = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)
    small = cv2.GaussianBlur(small, (0, 0), 3)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(th)
    if n > 1:
        k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        th = (lab == k).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filled = np.zeros_like(th)
    if cnts:
        cv2.drawContours(filled, cnts, -1, 255, -1)
    filled = cv2.erode(filled, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    return cv2.resize(filled, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST) > 0


def flatten(img: np.ndarray, mask: np.ndarray, sigma: float = 24.0) -> np.ndarray:
    """Divide by a smooth background estimated inside the disk (normalised convolution)."""
    s = 4  # work at 1/4 resolution for speed; the background is smooth anyway
    h, w = img.shape
    small = cv2.resize(img.astype(np.float32), (w // s, h // s), interpolation=cv2.INTER_AREA)
    m = cv2.resize(mask.astype(np.float32), (w // s, h // s), interpolation=cv2.INTER_AREA)
    # median filter first so dark filaments don't pull the background down
    med = cv2.medianBlur(np.clip(small, 0, 255).astype(np.uint8), 15).astype(np.float32)
    num = cv2.GaussianBlur(med * m, (0, 0), sigma / s * 2)
    den = cv2.GaussianBlur(m, (0, 0), sigma / s * 2)
    bg = num / np.maximum(den, 1e-3)
    bg = cv2.resize(bg, (w, h), interpolation=cv2.INTER_LINEAR)
    ratio = img.astype(np.float32) / np.maximum(bg, 1.0)
    ratio[~mask] = 1.0
    # map ratio 0.6..1.25 -> 0..255
    out = (ratio - 0.6) / (1.25 - 0.6) * 255.0
    return np.clip(out, 0, 255).astype(np.uint8)


def make_input(img: np.ndarray) -> np.ndarray:
    """Return (H, W, 3) uint8 network input at native resolution."""
    mask = disk_mask(img)
    vals = img[mask] if mask.any() else img.ravel()
    lo, hi = np.percentile(vals, [0.5, 99.5])
    ch0 = np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1.0) * 255.0, 0, 255).astype(np.uint8)
    ch1 = flatten(img, mask)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16, 16))
    ch2 = clahe.apply(ch1)
    ch2[~mask] = 0
    return np.dstack([ch0, ch1, ch2])


def load_input(path: str, size: int) -> np.ndarray:
    """Read an observation and return the 3-channel input resized to (size, size)."""
    x = make_input(read_gray(path))
    if size != x.shape[0]:
        x = cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA)
    return x

In [ ]:
%%writefile solarfil/rescore.py
"""Learned instance re-scoring (false-positive filter).

The segmentation network leaves many low-quality candidate instances (FP ~ 0.5 x TP on
validation). Each candidate is described by shape / confidence features (postprocess.FEATURES)
and a gradient-boosting regressor predicts the fraction of annotators that would accept it
(IoU > 0.5). Candidates below a threshold are dropped. Model and threshold are fitted on
out-of-fold predictions only, with photo-grouped cross-validation and the official PQ metric.

python -m solarfil.rescore --data_root <root> --oof <oof_dir> [...] --postcfg postcfg.json --out rescorer.pkl
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import pickle
from concurrent.futures import ProcessPoolExecutor
from dataclasses import replace

import numpy as np
from pycocotools import mask as mask_util
from sklearn.ensemble import HistGradientBoostingRegressor

from .annotations import load_photos, photo_gt_rles
from .metric import PQStats
from .postprocess import PostCfg, encode, instances, load_avg

_G = {}


def _candidates(stem):
    p = load_avg([os.path.join(d, f"{stem}.npz") for d in _G["dirs"]])
    masks, feats = instances(p[0], p[1], _G["cfg"], want_feats=True)
    rles = [{"size": [2048, 2048], "counts": r.encode()} for r in encode(masks)]
    ious = []  # per annotator: (n_pred, n_gt) IoU matrix
    for gt in _G["gts"][stem].values():
        if len(gt) == 0:
            continue
        ious.append(np.asarray(mask_util.iou(rles, gt, [0] * len(gt))).reshape(len(rles), len(gt))
                    if rles else np.zeros((0, len(gt))))
    return stem, np.asarray(feats, np.float32).reshape(-1, 17), ious


def pq_subset(items, keep_fn) -> PQStats:
    tot = PQStats()
    for feats, ious, score in items:
        k = keep_fn(score) if len(score) else np.zeros(0, bool)
        for m in ious:
            mm = m[k]
            hit = mm > 0.5
            tot.iou_sum += float(mm[hit].sum())
            tot.tp += int(hit.sum())
            tot.fp += int((hit.sum(axis=1) == 0).sum())
            tot.fn += int((hit.sum(axis=0) == 0).sum())
    return tot


def labels(ious):
    if not ious or ious[0].shape[0] == 0:
        return np.zeros(0)
    return np.mean([(m > 0.5).any(axis=1) for m in ious], axis=0)


def main(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True)
    a.add_argument("--oof", nargs="+", required=True)
    a.add_argument("--postcfg", required=True, help="base post-processing; min_score/min_area are relaxed")
    a.add_argument("--workers", type=int, default=4)
    a.add_argument("--folds", type=int, default=5)
    a.add_argument("--out", default="rescorer.pkl")
    args = a.parse_args(argv)

    base = PostCfg(**json.load(open(args.postcfg))["cfg"])
    cand_cfg = replace(base, min_score=min(base.min_score, 0.35), min_area=min(base.min_area, 100))
    photos = load_photos(os.path.join(args.data_root, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    stems = sorted(set.intersection(*[{os.path.basename(f)[:-4] for f in glob.glob(os.path.join(d, "*.npz"))}
                                      for d in args.oof]))
    _G.update(dirs=args.oof, cfg=cand_cfg, gts={s: photo_gt_rles(photos[s]) for s in stems})
    with ProcessPoolExecutor(args.workers, initializer=_G.update, initargs=(dict(_G),)) as pool:
        data = {s: (f, i) for s, f, i in pool.map(_candidates, stems, chunksize=2)}
    print(f"{len(stems)} photos, {sum(len(f) for f, _ in data.values())} candidates", flush=True)

    # baseline: the hand-tuned filter
    def base_keep(f):
        return (np.exp(f[:, 0]) >= base.min_area) & (f[:, 1] >= base.min_score)
    items = [(f, i, f) for f, i in data.values()]
    print("hand-tuned filter :", pq_subset(items, lambda f: base_keep(f) if len(f) else f), flush=True)

    # photo-grouped CV
    rng = np.random.default_rng(0)
    order = rng.permutation(stems)
    fold_of = {s: k % args.folds for k, s in enumerate(order)}
    oof_items = []
    params = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0)
    for k in range(args.folds):
        tr = [s for s in stems if fold_of[s] != k and len(data[s][0])]
        X = np.concatenate([data[s][0] for s in tr])
        y = np.concatenate([labels(data[s][1]) for s in tr])
        mdl = HistGradientBoostingRegressor(**params).fit(X, y)
        for s in stems:
            if fold_of[s] == k:
                f, i = data[s]
                oof_items.append((f, i, mdl.predict(f) if len(f) else np.zeros(0)))
    best_t, best = None, None
    for t in np.arange(0.10, 0.71, 0.025):
        st = pq_subset(oof_items, lambda sc, t=t: sc >= t)
        if best is None or st.pq > best.pq:
            best_t, best = float(t), st
    print(f"learned rescoring (CV): thr={best_t:.3f} {best}", flush=True)

    X = np.concatenate([data[s][0] for s in stems if len(data[s][0])])
    y = np.concatenate([labels(data[s][1]) for s in stems if len(data[s][0])])
    mdl = HistGradientBoostingRegressor(**params).fit(X, y)
    with open(args.out, "wb") as f:
        pickle.dump({"model": mdl, "thr": best_t, "cand_cfg": cand_cfg.as_dict(), "cv_pq": best.pq}, f)
    print("saved", args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile solarfil/train.py
"""Train one fold.

python -m solarfil.train --data_root <MAGFiLO_1.0_Kaggle_2026> --work /kaggle/working --fold 0
"""
from __future__ import annotations

import argparse
import json
import math
import os
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

from .annotations import load_photos, photo_gt_rles
from .data import CropDataset, build_cache, load_full, make_folds
from .metric import PQStats, photo_stats
from .model import EMA, FilamentLoss, FilamentNet, predict_full
from .postprocess import PostCfg, instances_rle


def get_args(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True)
    a.add_argument("--work", default="work")
    a.add_argument("--fold", type=int, default=0)
    a.add_argument("--n_folds", type=int, default=5)
    a.add_argument("--size", type=int, default=1536, help="full-disk working resolution")
    a.add_argument("--crop", type=int, default=768)
    a.add_argument("--arch", default="Unet")
    a.add_argument("--encoder", default="efficientnet-b4")
    a.add_argument("--weights", default="imagenet")
    a.add_argument("--epochs", type=int, default=40)
    a.add_argument("--bs", type=int, default=6)
    a.add_argument("--lr", type=float, default=6e-4)
    a.add_argument("--wd", type=float, default=1e-4)
    a.add_argument("--samples_per_epoch", type=int, default=0, help="0 -> 3 crops per training photo")
    a.add_argument("--border_weight", type=float, default=0.5)
    a.add_argument("--val_every", type=int, default=4)
    a.add_argument("--val_tta", type=int, default=2)
    a.add_argument("--workers", type=int, default=4)
    a.add_argument("--limit", type=int, default=0, help="debug: use only N photos")
    a.add_argument("--ema", type=float, default=0.998)
    a.add_argument("--cache_root", default="", help="where to put the image cache (default: --work)")
    a.add_argument("--seed", type=int, default=42)
    return a.parse_args(argv)


def evaluate(model, stems, cache_dir, gts, cfg: PostCfg, tta: int) -> PQStats:
    total = PQStats()
    for s in stems:
        p = predict_full(model, load_full(s, cache_dir), tta=tta).numpy()
        total.add(photo_stats(gts[s], instances_rle(p[0], p[1], cfg)))
    return total


def main(argv=None):
    args = get_args(argv)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    json_path = os.path.join(args.data_root, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
    img_dir = os.path.join(args.data_root, "train", "train_images")
    photos = load_photos(json_path)
    photos = {k: v for k, v in photos.items() if os.path.exists(os.path.join(img_dir, v.file_name))}
    if args.limit:
        photos = dict(list(photos.items())[: args.limit])
    cache_dir = os.path.join(args.cache_root or args.work, f"cache_{args.size}")
    print(f"photos={len(photos)} device={dev} cache={cache_dir}", flush=True)
    build_cache(photos, img_dir, cache_dir, args.size, workers=args.workers)

    folds = make_folds(list(photos), args.n_folds, args.seed)
    folds.to_csv(os.path.join(args.work, "folds.csv"), index=False)
    tr = folds.loc[folds.fold != args.fold, "stem"].tolist()
    va = folds.loc[folds.fold == args.fold, "stem"].tolist()
    gts = {s: photo_gt_rles(photos[s]) for s in va}
    print(f"fold {args.fold}: train={len(tr)} val={len(va)}", flush=True)

    spe = args.samples_per_epoch or 3 * len(tr)
    ds = CropDataset(tr, cache_dir, crop=args.crop, samples_per_epoch=spe, seed=args.seed)
    dl = DataLoader(ds, batch_size=args.bs, shuffle=True, num_workers=args.workers, drop_last=True,
                    pin_memory=dev.type == "cuda", persistent_workers=args.workers > 0)

    model = FilamentNet(args.arch, args.encoder, None if args.weights == "none" else args.weights).to(dev)
    ema = EMA(model, decay=args.ema)
    crit = FilamentLoss(border_weight=args.border_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
    steps = args.epochs * len(dl)
    warm = min(300, steps // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))) * 0.98 + 0.02)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    tag = f"{args.arch}_{args.encoder}_s{args.size}_f{args.fold}"
    ckpt = os.path.join(args.work, f"{tag}.pth")
    best, log = -1.0, []
    cfg = PostCfg()
    for ep in range(1, args.epochs + 1):
        model.train()
        t0, tot = time.time(), 0.0
        for x, y in dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                out = model(x)
            loss = crit(out.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update(model)
            tot += loss.item()
        msg = f"ep {ep:02d} loss {tot / len(dl):.4f} lr {sched.get_last_lr()[0]:.2e} {time.time() - t0:.0f}s"
        if ep % args.val_every == 0 or ep == args.epochs:
            st = evaluate(ema.ema, va, cache_dir, gts, cfg, args.val_tta)
            msg += f" | val {st}"
            log.append({"epoch": ep, "pq": st.pq, "sq": st.sq, "dq": st.dq})
            if st.pq > best:
                best = st.pq
                torch.save({"state_dict": ema.ema.state_dict(), "args": vars(args), "epoch": ep, "pq": best}, ckpt)
                msg += " *"
        print(msg, flush=True)

    # out-of-fold probabilities of the best checkpoint (for post-processing tuning)
    sd = torch.load(ckpt, map_location=dev)["state_dict"]
    model.load_state_dict(sd)
    model.eval()
    oof_dir = os.path.join(args.work, f"oof_{args.arch}_{args.encoder}_s{args.size}")
    os.makedirs(oof_dir, exist_ok=True)
    for s in va:
        p = predict_full(model, load_full(s, cache_dir), tta=8).numpy()
        np.savez_compressed(os.path.join(oof_dir, f"{s}.npz"), p=np.round(p * 255).astype(np.uint8))
    with open(os.path.join(args.work, f"{tag}_log.json"), "w") as f:
        json.dump({"best_pq": best, "log": log, "args": vars(args)}, f, indent=1)
    print(f"done fold {args.fold}: best val PQ {best:.4f} -> {ckpt}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile solarfil/tune.py
"""Tune post-processing on out-of-fold probabilities with the official PQ metric.

python -m solarfil.tune --data_root <root> --oof work/oof_Unet_efficientnet-b4_s1536 [--oof other_dir ...]
Multiple --oof dirs are averaged (model ensemble).
"""
from __future__ import annotations

import argparse
import glob
import json
import os
from concurrent.futures import ProcessPoolExecutor
from dataclasses import replace

import numpy as np

from .annotations import load_photos, photo_gt_rles
from .metric import PQStats, photo_stats
from .postprocess import PostCfg, instances_rle, load_avg

_G = {}


def _load(stem):
    return load_avg([os.path.join(d, f"{stem}.npz") for d in _G["dirs"]])


def _score_one(args):
    stem, cfg = args
    p = _load(stem)
    return photo_stats(_G["gts"][stem], instances_rle(p[0], p[1], cfg))


_CACHE = {}


def score(stems, cfg, pool) -> PQStats:
    key = json.dumps(cfg.as_dict(), sort_keys=True)
    if key in _CACHE:
        return PQStats(**_CACHE[key])
    tot = PQStats()
    for s in pool.map(_score_one, [(st, cfg) for st in stems], chunksize=2):
        tot.add(s)
    _CACHE[key] = dict(iou_sum=tot.iou_sum, tp=tot.tp, fp=tot.fp, fn=tot.fn)
    if _G.get("cache_file"):
        with open(_G["cache_file"], "a") as f:
            f.write(json.dumps({"k": key, "v": _CACHE[key]}) + "\n")
    return tot


GRID = {
    "thr": [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60],
    "link_px": [0, 4, 8, 14, 22, 32],
    "border_thr": [1.01, 0.7, 0.5, 0.35],
    "min_area": [50, 100, 150, 200, 300, 450, 650],
    "min_score": [0.40, 0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85],
    "close_px": [0, 2, 4],
    "seed_thr": [1.01, 0.9, 0.8, 0.7, 0.6],
    "min_cc": [10, 40, 100],
    "fill_holes": [True, False],
}


def main(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True)
    a.add_argument("--oof", nargs="+", required=True)
    a.add_argument("--max_photos", type=int, default=0)
    a.add_argument("--rounds", type=int, default=2)
    a.add_argument("--workers", type=int, default=4)
    a.add_argument("--out", default="postcfg.json")
    a.add_argument("--init", default="", help="start from this postcfg.json")
    a.add_argument("--cache", default="", help="jsonl of already-scored configs (makes the run resumable)")
    args = a.parse_args(argv)

    photos = load_photos(os.path.join(args.data_root, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    stems = sorted(set.intersection(*[{os.path.basename(f)[:-4] for f in glob.glob(os.path.join(d, "*.npz"))}
                                      for d in args.oof]))
    if args.max_photos:
        rng = np.random.default_rng(0)
        stems = sorted(rng.choice(stems, size=min(args.max_photos, len(stems)), replace=False).tolist())
    _G["dirs"] = args.oof
    _G["gts"] = {s: photo_gt_rles(photos[s]) for s in stems}
    print(f"tuning on {len(stems)} OOF photos", flush=True)

    cfg = PostCfg(**json.load(open(args.init))["cfg"]) if args.init else PostCfg()
    if args.cache:
        _G["cache_file"] = args.cache
        if os.path.exists(args.cache):
            for line in open(args.cache):
                r = json.loads(line)
                _CACHE[r["k"]] = r["v"]
            print(f"resumed {len(_CACHE)} cached configs", flush=True)
    with ProcessPoolExecutor(args.workers, initializer=_G.update, initargs=(dict(_G),)) as pool:
        best = score(stems, cfg, pool)
        print("start", cfg, best, flush=True)
        for r in range(args.rounds):
            for k, vals in GRID.items():
                for v in vals:
                    if getattr(cfg, k) == v:
                        continue
                    c = replace(cfg, **{k: v})
                    st = score(stems, c, pool)
                    flag = ""
                    if st.pq > best.pq + 1e-5:
                        cfg, best, flag = c, st, "  <-- best"
                    print(f"r{r} {k}={v}: {st}{flag}", flush=True)
            print(f"after round {r}: {cfg} {best}", flush=True)
    with open(args.out, "w") as f:
        json.dump({"cfg": cfg.as_dict(), "pq": best.pq, "sq": best.sq, "dq": best.dq,
                   "n_photos": len(stems), "oof": args.oof}, f, indent=1)
    print("saved", args.out)


if __name__ == "__main__":
    main()

In [ ]:
open('solarfil5/__init__.py','w').close()

In [ ]:
%%writefile solarfil5/decode.py
"""Spine/offset instance decoder: instances = connected spines (linked only through foreground support),
foreground pixels are assigned by the predicted centre-line offset vote. No isotropic dilation of the mask."""
from __future__ import annotations
from dataclasses import dataclass, asdict
import cv2
import numpy as np
from scipy import ndimage as ndi
from skimage.morphology import skeletonize
from pycocotools import mask as mask_util

NATIVE = 2048
OFF_SCALE = 32.0


@dataclass
class DecCfg:
    thr: float = 0.40          # foreground
    spine_thr: float = 0.40    # spine heat-map seed threshold
    gap_px: int = 12           # max spine gap (working px) bridged, only through fg > thr_link
    thr_link: float = 0.25
    border_thr: float = 1.01   # >1 disables
    min_spine: int = 6         # skeleton px
    min_area: int = 250        # native px^2
    min_score: float = 0.5
    max_assign: int = 40       # max distance (working px) for nearest-spine fallback

    def as_dict(self): return asdict(self)


def decode(fg, spine, border, off, cfg: DecCfg, native_scale: float | None = None):
    """fg/spine/border: (S,S) probs; off: (2,S,S) in [-1,1] (dy,dx)/OFF_SCALE. Returns list of (S,S) uint8 masks."""
    S = fg.shape[0]
    sc = (S / NATIVE) if native_scale is None else native_scale
    fgb = fg > cfg.thr
    sk = skeletonize((spine > cfg.spine_thr) & (fg > cfg.thr_link))
    if cfg.border_thr <= 1.0:
        sk &= ~(border > cfg.border_thr)
    # link spine fragments: close small gaps, geodesically (through fg support) -> components
    r = max(cfg.gap_px // 2, 1)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    linked = (cv2.dilate(sk.astype(np.uint8), k) > 0) & (fg > cfg.thr_link)
    ncc, lab_l = cv2.connectedComponents(linked.astype(np.uint8), connectivity=8)
    seeds = np.where(sk, lab_l, 0).astype(np.int32)
    # drop tiny spines
    cnt = np.bincount(seeds.ravel(), minlength=ncc)
    seeds[np.isin(seeds, np.flatnonzero(cnt < cfg.min_spine))] = 0
    if not seeds.any():
        return []
    # (a) offset vote for every fg pixel
    ys, xs = np.nonzero(fgb)
    vy = np.clip(np.round(ys + off[0][ys, xs] * OFF_SCALE).astype(int), 0, S - 1)
    vx = np.clip(np.round(xs + off[1][ys, xs] * OFF_SCALE).astype(int), 0, S - 1)
    # voted location -> nearest seed label (spine is thin; look up through the nearest-seed map)
    dist, (iy, ix) = ndi.distance_transform_edt(seeds == 0, return_indices=True)
    lab_near = seeds[iy, ix]
    vote = lab_near[vy, vx]
    ok_vote = dist[vy, vx] <= 3
    own = lab_near[ys, xs]
    use = np.where(ok_vote, vote, np.where(dist[ys, xs] <= cfg.max_assign, own, 0))
    lab = np.zeros((S, S), np.int32)
    lab[ys, xs] = use
    # connected pixels only: keep the component of each label that touches its spine
    out = []
    objs = ndi.find_objects(lab)
    for i, sl in enumerate(objs, 1):
        if sl is None:
            continue
        m = lab[sl] == i
        m = ndi.binary_fill_holes(m | (seeds[sl] == i))
        m &= (lab[sl] == i) | (lab[sl] == 0)
        if m.sum() / (sc * sc) < cfg.min_area:
            continue
        if float(fg[sl][m].mean()) < cfg.min_score:
            continue
        full = np.zeros((S, S), np.uint8)
        full[sl][m] = 1
        out.append(full)
    return out


def to_native_rle(masks, S):
    res = []
    for m in masks:
        if S != NATIVE:
            m = cv2.resize(m, (NATIVE, NATIVE), interpolation=cv2.INTER_NEAREST)
        res.append(mask_util.encode(np.asfortranarray(m))["counts"].decode())
    return res

In [ ]:
%%writefile solarfil5/folds.py
"""Time-block GroupKFold: a filament drifts across the disk for ~1 solar rotation (27 d), so all photos
inside one block of `block_days` days (+ optional embargo) stay in the same fold."""
from __future__ import annotations
import datetime as dt
import numpy as np
import pandas as pd


def stem_date(stem: str) -> dt.date:
    return dt.date(int(stem[:4]), int(stem[4:6]), int(stem[6:8]))


def make_block_folds(stems: list[str], n_folds: int = 5, block_days: int = 27, seed: int = 42) -> pd.DataFrame:
    df = pd.DataFrame({"stem": sorted(stems)})
    d0 = min(stem_date(s) for s in df.stem)
    df["day"] = [(stem_date(s) - d0).days for s in df.stem]
    df["block"] = df["day"] // block_days
    # greedy balanced assignment of blocks to folds (largest block first), deterministic given seed
    sizes = df.groupby("block").size().sort_values(ascending=False)
    rng = np.random.default_rng(seed)
    order = sizes.index.to_numpy()
    load = np.zeros(n_folds)
    fold_of = {}
    for b in order:
        cand = np.flatnonzero(load == load.min())
        f = int(rng.choice(cand))
        fold_of[b] = f
        load[f] += sizes[b]
    df["fold"] = df["block"].map(fold_of)
    return df


def leakage_report(df: pd.DataFrame, gap_days: int = 3) -> dict:
    """Count val photos having a train photo within `gap_days` days (cross-fold temporal neighbours)."""
    out = {}
    for f in sorted(df.fold.unique()):
        va = df.loc[df.fold == f, "day"].to_numpy()
        tr = np.sort(df.loc[df.fold != f, "day"].to_numpy())
        idx = np.searchsorted(tr, va)
        near = 0
        for v, i in zip(va, idx):
            c = [tr[j] for j in (i - 1, i) if 0 <= j < len(tr)]
            near += any(abs(v - x) <= gap_days for x in c)
        out[f] = (len(va), near)
    return out

In [ ]:
%%writefile solarfil5/infer5.py
"""Fast GPU inference: FP16 autocast + channels_last, full dihedral TTA (8), all models on the same batch, Fortran RLE.

python -m solarfil5.infer5 --data_root <root> --ckpts work/*.pth --postcfg dec.json --out submission.csv
"""
from __future__ import annotations
import argparse, glob, json, os, time
import numpy as np, pandas as pd, torch
from pycocotools import mask as mask_util
from .model5 import Net5, predict
from .decode import decode, DecCfg, to_native_rle


def main(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True); a.add_argument("--ckpts", nargs="+", required=True)
    a.add_argument("--postcfg", default=""); a.add_argument("--out", default="submission.csv"); a.add_argument("--tta", type=int, default=8)
    a.add_argument("--work", default="work"); a.add_argument("--save_probs", default="")
    args = a.parse_args(argv)
    from solarfil.preprocess import load_input
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = DecCfg(**json.load(open(args.postcfg))["cfg"]) if args.postcfg else DecCfg()
    groups = {}
    for p in args.ckpts:
        ck = torch.load(p, map_location="cpu", weights_only=False); ar = ck["args"]
        m = Net5(ar["encoder"], None); m.load_state_dict(ck["state_dict"])
        groups.setdefault(ar["size"], []).append(m.to(dev).eval().to(memory_format=torch.channels_last))
    paths = sorted(glob.glob(os.path.join(args.data_root, "test", "test_images", "*.jp*g")))
    rows, t0 = [], time.time()
    top = max(groups)
    for path in paths:
        stem = os.path.splitext(os.path.basename(path))[0]
        acc, n = None, 0
        for size, models in groups.items():
            x = torch.from_numpy(load_input(path, size).transpose(2, 0, 1).astype(np.float32) / 255.0)[None].to(dev)
            p = predict(models, x, tta=args.tta)
            if size != top:
                p = torch.nn.functional.interpolate(p, size=(top, top), mode="bilinear", align_corners=False)
            acc = p * len(models) if acc is None else acc + p * len(models); n += len(models)
        p = (acc / n)[0].cpu().numpy()
        if args.save_probs:
            os.makedirs(args.save_probs, exist_ok=True); np.savez_compressed(os.path.join(args.save_probs, f"{stem}.npz"), p=p.astype(np.float16))
        for i, r in enumerate(to_native_rle(decode(p[0], p[2], p[1], p[3:5], cfg), top), 1):
            rows.append({"filament_id": f"{stem}_{i}", "segmentation_rle": r})
    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    assert df.filament_id.is_unique
    for stem, g in df.groupby(df.filament_id.str.split("_", n=1).str[0]):
        rl = [{"size": [2048, 2048], "counts": r.encode()} for r in g.segmentation_rle]
        if len(rl) > 1: assert mask_util.area(mask_util.merge(rl, intersect=False)) == mask_util.area(rl).sum(), f"overlap in {stem}"
    df.to_csv(args.out, index=False); print(f"wrote {args.out}: {len(df)} filaments, {time.time() - t0:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile solarfil5/model5.py
"""3-head network: fg | border | spine heat-map | centre-line offset (dy,dx). One Unet, 5 output channels."""
from __future__ import annotations
import torch, torch.nn as nn, torch.nn.functional as F
import segmentation_models_pytorch as smp

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class Net5(nn.Module):
    def __init__(self, encoder="efficientnet-b4", weights="imagenet"):
        super().__init__()
        self.net = smp.Unet(encoder_name=encoder, encoder_weights=weights, in_channels=3, classes=5,
                            decoder_attention_type="scse")
        self.register_buffer("mean", MEAN.clone(), persistent=False)
        self.register_buffer("std", STD.clone(), persistent=False)

    def forward(self, x):                      # raw logits (B,5,H,W); offsets are tanh'd in the loss / predict
        return self.net((x - self.mean) / self.std)


def dice(logit, t, w=None, eps=1.0):
    p = torch.sigmoid(logit)
    if w is not None:
        p, t = p * w, t * w
    i = (p * t).sum((1, 2)); return (1 - (2 * i + eps) / (p.sum((1, 2)) + t.sum((1, 2)) + eps)).mean()


class Loss5(nn.Module):
    """y: (B,6,H,W) in [0,1]: fg, border, spine, offy, offx (0.5-centred), weight/4.
    Instance-size inverse weighting: w = sqrt(median_area/area) (== 1/sqrt(area) up to a constant), baked into y[:,5]."""
    def __init__(self, pos_w=3.0, w_border=0.5, w_spine=1.0, w_off=2.0):
        super().__init__(); self.pw, self.wb, self.ws, self.wo = pos_w, w_border, w_spine, w_off

    def forward(self, out, y):
        fg, bd, sp = y[:, 0], y[:, 1], y[:, 2]
        off = (y[:, 3:5] - 0.5) * 2.0
        w = (y[:, 5] * 4.0).clamp(min=0.5)                  # per-pixel instance weight (1 outside filaments)
        w_fg = torch.where(fg > 0.5, w, torch.ones_like(w))
        pw = torch.tensor(self.pw, device=out.device)
        bce = (F.binary_cross_entropy_with_logits(out[:, 0], fg, pos_weight=pw, reduction="none") * w_fg).mean()
        l_fg = bce + dice(out[:, 0], fg, None)
        l_bd = F.binary_cross_entropy_with_logits(out[:, 1], bd, pos_weight=pw * 3) + 0.5 * dice(out[:, 1], bd)
        l_sp = (F.binary_cross_entropy_with_logits(out[:, 2], sp, pos_weight=pw * 3, reduction="none") * w_fg).mean() \
            + dice(out[:, 2], sp)
        m = (fg > 0.5).float()
        pred_off = torch.tanh(out[:, 3:5])
        l_off = (F.smooth_l1_loss(pred_off, off, reduction="none", beta=0.1).sum(1) * m * w).sum() / (m.sum() + 1.0)
        return l_fg + self.wb * l_bd + self.ws * l_sp + self.wo * l_off


@torch.no_grad()
def predict(models, x, tta=8, amp=True):
    """x (B,3,H,W) in [0,1] on device. Full dihedral TTA (8). Offsets are rotated back as vectors.
    returns probs (B,5,H,W): fg, border, spine (sigmoid) and offy, offx (tanh)."""
    dev = x.device
    acc = None
    for k in range(tta):
        r, f = k % 4, k >= 4
        xt = torch.rot90(x, r, dims=(2, 3))
        if f: xt = torch.flip(xt, dims=(3,))
        for m in models:
            with torch.autocast(device_type=dev.type, enabled=amp and dev.type == "cuda", dtype=torch.float16):
                o = m(xt)
            o = o.float()
            p = torch.cat([torch.sigmoid(o[:, :3]), torch.tanh(o[:, 3:5])], 1)
            if f:                                  # undo flip along W: dx -> -dx
                p = torch.flip(p, dims=(3,)); p[:, 4] = -p[:, 4]
            # undo rot90 by r quarter turns (counter-clockwise by r): vector (dy,dx) rotates accordingly
            p = torch.rot90(p, -r, dims=(2, 3))
            dy, dx = p[:, 3].clone(), p[:, 4].clone()
            for _ in range(r):                     # inverse of a CCW quarter turn: (dy,dx) -> (-dx,dy)... applied r times
                dy, dx = dx.clone(), -dy.clone()
            p[:, 3], p[:, 4] = dy, dx
            acc = p if acc is None else acc + p
    return acc / (tta * len(models))

In [ ]:
%%writefile solarfil5/targets.py
"""Instance-level targets built from the COCO polygons + `spine` polylines.

Per annotator copy -> (fg, border, spine heat-map, centre-line offset (dy,dx), instance-size weight, instance id map).
Instance-level consensus (IoU>0.5 matches across annotators) is provided by `consensus_photo`.
"""
from __future__ import annotations
import cv2
import numpy as np
from pycocotools import mask as mask_util
from scipy import ndimage as ndi

NATIVE = 2048
OFF_SCALE = 32.0          # offsets are stored as offset/OFF_SCALE in [-1,1] (px at working size)
MEDIAN_AREA = 1228.0      # native px^2 (train median)


def load_instances(json_path: str):
    """stem -> {annotator_id: [ {poly, spine}, ... ]}"""
    import json, os
    from collections import defaultdict
    coco = json.load(open(json_path, encoding="utf-8"))
    anns = defaultdict(list)
    for a in coco["annotations"]:
        anns[a["image_id"]].append(a)
    out = {}
    for im in coco["images"]:
        stem = os.path.splitext(im["file_name"])[0]
        d = out.setdefault(stem, {})
        d[im["id"]] = [{"poly": a["segmentation"][0], "spine": a["spine"]} for a in anns.get(im["id"], [])
                       if a["segmentation"] and len(a["segmentation"][0]) >= 6]
    return out


def _mask(poly, size):
    s = size / NATIVE
    p = (np.asarray(poly, np.float64) * s).reshape(-1, 2)
    m = np.zeros((size, size), np.uint8)
    cv2.fillPoly(m, [np.round(p).astype(np.int32)], 1)
    return m


def _spine_mask(spine, size, thick=1):
    s = size / NATIVE
    p = (np.asarray(spine, np.float64) * s).reshape(-1, 2)
    m = np.zeros((size, size), np.uint8)
    if len(p) >= 2:
        cv2.polylines(m, [np.round(p).astype(np.int32)], False, 1, thickness=thick)
    else:
        x, y = np.round(p[0]).astype(int); m[np.clip(y, 0, size - 1), np.clip(x, 0, size - 1)] = 1
    return m


def annotator_targets(insts: list[dict], size: int, border_px: int = 5, sigma: float = 3.0):
    """insts: list of {poly, spine}. Returns dict of float32/int32 arrays at (size,size)."""
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * border_px + 1, 2 * border_px + 1))
    fg = np.zeros((size, size), np.uint8)
    cnt = np.zeros((size, size), np.uint8)
    spine = np.zeros((size, size), np.uint8)
    iid = np.zeros((size, size), np.int32)
    off = np.zeros((2, size, size), np.float32)
    w = np.ones((size, size), np.float32)
    sc = size / NATIVE
    for i, it in enumerate(insts, 1):
        m = _mask(it["poly"], size)
        sp = _spine_mask(it["spine"], size)
        sp &= (cv2.dilate(m, np.ones((3, 3), np.uint8)) > 0).astype(np.uint8)   # keep spine inside its own filament
        if not sp.any():
            ys, xs = np.nonzero(m)
            if len(ys) == 0:
                continue
            sp[ys[len(ys) // 2], xs[len(xs) // 2]] = 1
        ys, xs = np.nonzero(m)
        if len(ys) == 0:
            continue
        y0, y1, x0, x1 = max(ys.min() - 2, 0), min(ys.max() + 3, size), max(xs.min() - 2, 0), min(xs.max() + 3, size)
        mc, sc_ = m[y0:y1, x0:x1], sp[y0:y1, x0:x1]
        d, (iy, ix) = ndi.distance_transform_edt(sc_ == 0, return_indices=True)
        gy, gx = np.mgrid[y0:y1, x0:x1]
        sel = (mc > 0) & (iid[y0:y1, x0:x1] == 0)
        oy = (iy + y0 - gy).astype(np.float32); ox = (ix + x0 - gx).astype(np.float32)
        off[0, y0:y1, x0:x1][sel] = oy[sel] / OFF_SCALE
        off[1, y0:y1, x0:x1][sel] = ox[sel] / OFF_SCALE
        iid[y0:y1, x0:x1][sel] = i
        area_native = m.sum() / (sc * sc)
        w[y0:y1, x0:x1][sel] = np.clip(np.sqrt(MEDIAN_AREA / max(area_native, 1.0)), 0.5, 4.0)   # 1/sqrt(area)
        fg |= m
        spine |= sp
        if len(insts) > 1:
            cnt += cv2.dilate(m, k)
    heat = cv2.GaussianBlur(spine.astype(np.float32), (0, 0), sigma * max(sc, 0.25))
    heat = heat / max(heat.max(), 1e-6) if heat.max() > 0 else heat
    heat = np.maximum(heat, spine.astype(np.float32))
    return {"fg": fg.astype(np.float32), "border": (cnt >= 2).astype(np.float32), "spine": heat.astype(np.float32),
            "off": np.clip(off, -1, 1), "w": w, "iid": iid}


def pack(t: dict) -> np.ndarray:
    """uint8 (6,S,S) cache layout: fg, border, spine, offy, offx, weight."""
    u = lambda a: np.round(np.clip(a, 0, 1) * 255).astype(np.uint8)
    o = lambda a: np.round((np.clip(a, -1, 1) * 0.5 + 0.5) * 255).astype(np.uint8)
    return np.stack([u(t["fg"]), u(t["border"]), u(t["spine"]), o(t["off"][0]), o(t["off"][1]),
                     np.round(np.clip(t["w"] / 4.0, 0, 1) * 255).astype(np.uint8)])


def _iou(a, b):
    u = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / u if u else 0.0


def consensus_photo(annotators: dict, size: int, min_votes: int = 2):
    """Instance-level consensus over annotator copies.
    Instances matched across annotators (IoU>0.5, greedy against a reference copy) are merged to their
    pixel-majority mask; their spine = spine of the reference copy member with highest mean IoU.
    Unmatched instances are kept only if the photo has a single annotator or they reach `min_votes`."""
    ids = list(annotators)
    if len(ids) == 1:
        return annotators[ids[0]]
    masks = {a: [_mask(i["poly"], size) for i in annotators[a]] for a in ids}
    ref = max(ids, key=lambda a: len(masks[a]))
    out = []
    for r, rm in enumerate(masks[ref]):
        group, best_spine = [rm], annotators[ref][r]
        for a in ids:
            if a == ref:
                continue
            j = max(range(len(masks[a])), key=lambda j: _iou(rm, masks[a][j]), default=None)
            if j is not None and _iou(rm, masks[a][j]) > 0.5:
                group.append(masks[a][j])
        if len(group) >= min_votes:
            maj = (np.mean(group, 0) >= 0.5).astype(np.uint8)
            cs, _ = cv2.findContours(maj, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not cs:
                continue
            c = max(cs, key=cv2.contourArea).reshape(-1, 2) * (NATIVE / size)
            if len(c) >= 3:
                out.append({"poly": c.reshape(-1).tolist(), "spine": best_spine["spine"]})
    return out

In [ ]:
%%writefile solarfil5/train5.py
"""v5 training: time-block CV, instance-level targets (spine, offset, size weight), per-annotator sampling.

python -m solarfil5.train5 --data_root <MAGFiLO_1.0_Kaggle_2026> --work out --fold 0 [--size 1536] [--epochs 36]
Needs solarfil (v2) for preprocessing/metric/old post-processing helpers on PYTHONPATH.
"""
from __future__ import annotations
import argparse, json, math, os, random, time
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from .folds import make_block_folds
from .targets import load_instances, annotator_targets, consensus_photo, pack
from .model5 import Net5, Loss5, predict


def build_cache(ins, img_dir, files, cache, size, workers=4):
    from solarfil.preprocess import load_input
    os.makedirs(cache, exist_ok=True)
    for stem, fn in files.items():
        xp = os.path.join(cache, f"{stem}_x.npy")
        if os.path.exists(xp): continue
        np.save(xp, load_input(os.path.join(img_dir, fn), size))
        ann = ins[stem]
        ys = [pack(annotator_targets(v, size)) for v in list(ann.values())[:3] if len(v)]
        if not ys: ys = [pack(annotator_targets([], size))]
        np.save(os.path.join(cache, f"{stem}_y.npy"), np.stack(ys))          # (A,6,S,S) one entry per annotator copy


def dihedral_vec(x, y, k):
    """x (S,S,3), y (6,S,S); rotate/flip both and rotate the offset vector components accordingly."""
    r, f = k % 4, k >= 4
    if f:
        x = x[:, ::-1]; y = y[:, :, ::-1].copy(); y[4] = 255 - y[4]            # dx -> -dx  (0.5-centred uint8)
    x = np.rot90(x, r, axes=(0, 1)); y = np.rot90(y, r, axes=(1, 2)).copy()
    for _ in range(r):                                                         # (dy,dx) -> (-dx,dy)
        a, b = y[3].copy(), y[4].copy(); y[3], y[4] = 255 - b, a
    return np.ascontiguousarray(x), np.ascontiguousarray(y)


class Crops(Dataset):
    def __init__(self, stems, cache, crop=768, n=0, pos_frac=0.7, seed=0):
        self.stems, self.cache, self.crop, self.pos, self.seed = list(stems), cache, crop, pos_frac, seed
        self.n = n or 3 * len(self.stems)

    def __len__(self): return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed * 1_000_003 + i + random.getrandbits(31)) % 2**32)
        s = self.stems[rng.integers(len(self.stems))]
        x = np.load(os.path.join(self.cache, f"{s}_x.npy"), mmap_mode="r")
        Y = np.load(os.path.join(self.cache, f"{s}_y.npy"), mmap_mode="r")
        y = np.asarray(Y[rng.integers(Y.shape[0])])                          # one annotator copy per sample
        S, c = x.shape[0], self.crop
        if rng.random() < self.pos:
            ys, xs = np.nonzero(y[0, ::8, ::8] > 128)
            if len(ys):
                j = rng.integers(len(ys)); cy = ys[j] * 8 + rng.integers(-c // 3, c // 3 + 1); cx = xs[j] * 8 + rng.integers(-c // 3, c // 3 + 1)
            else: cy, cx = rng.integers(c // 2, S - c // 2 + 1, size=2)
        else: cy, cx = rng.integers(c // 2, S - c // 2 + 1, size=2)
        y0 = int(np.clip(cy - c // 2, 0, S - c)); x0 = int(np.clip(cx - c // 2, 0, S - c))
        xc, yc = np.asarray(x[y0:y0 + c, x0:x0 + c]), y[:, y0:y0 + c, x0:x0 + c]
        xc, yc = dihedral_vec(xc, yc, int(rng.integers(8)))
        from solarfil.data import photometric
        xc = photometric(np.ascontiguousarray(xc), rng)
        return torch.from_numpy(xc.transpose(2, 0, 1).copy()), torch.from_numpy(yc.astype(np.float32) / 255.0)


def main(argv=None):
    a = argparse.ArgumentParser()
    a.add_argument("--data_root", required=True); a.add_argument("--work", default="work"); a.add_argument("--fold", type=int, default=0)
    a.add_argument("--n_folds", type=int, default=5); a.add_argument("--block_days", type=int, default=27)
    a.add_argument("--size", type=int, default=1536); a.add_argument("--crop", type=int, default=768)
    a.add_argument("--encoder", default="efficientnet-b4"); a.add_argument("--weights", default="imagenet")
    a.add_argument("--epochs", type=int, default=36); a.add_argument("--bs", type=int, default=6)
    a.add_argument("--lr", type=float, default=6e-4); a.add_argument("--wd", type=float, default=1e-4)
    a.add_argument("--ema", type=float, default=0.998); a.add_argument("--workers", type=int, default=4)
    a.add_argument("--val_every", type=int, default=4); a.add_argument("--limit", type=int, default=0)
    a.add_argument("--cache_root", default=""); a.add_argument("--seed", type=int, default=42)
    args = a.parse_args(argv)
    torch.manual_seed(args.seed); np.random.seed(args.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    J = os.path.join(args.data_root, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
    img_dir = os.path.join(args.data_root, "train", "train_images")
    coco = json.load(open(J, encoding="utf-8"))
    files = {os.path.splitext(i["file_name"])[0]: i["file_name"] for i in coco["images"]}
    files = {k: v for k, v in files.items() if os.path.exists(os.path.join(img_dir, v))}
    if args.limit: files = dict(list(files.items())[: args.limit])
    ins = load_instances(J)
    cache = os.path.join(args.cache_root or args.work, f"cache5_{args.size}")
    os.makedirs(args.work, exist_ok=True)
    build_cache(ins, img_dir, files, cache, args.size)
    folds = make_block_folds(list(files), args.n_folds, args.block_days, args.seed); folds.to_csv(os.path.join(args.work, "folds_block.csv"), index=False)
    tr = folds.loc[folds.fold != args.fold, "stem"].tolist(); va = folds.loc[folds.fold == args.fold, "stem"].tolist()
    print(f"fold {args.fold}: train={len(tr)} val={len(va)} (block={args.block_days}d)", flush=True)
    dl = DataLoader(Crops(tr, cache, args.crop, seed=args.seed), batch_size=args.bs, shuffle=True, num_workers=args.workers,
                    drop_last=True, pin_memory=dev.type == "cuda", persistent_workers=args.workers > 0)
    model = Net5(args.encoder, None if args.weights == "none" else args.weights).to(dev)
    import copy
    ema = copy.deepcopy(model).eval()
    for p in ema.parameters(): p.requires_grad_(False)
    crit = Loss5(); opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
    steps = args.epochs * len(dl); warm = min(300, steps // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))) * 0.98 + 0.02)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")
    tag = f"Net5_{args.encoder}_s{args.size}_f{args.fold}"; ckpt = os.path.join(args.work, tag + ".pth"); best, log = -1.0, []
    for ep in range(1, args.epochs + 1):
        model.train(); t0, tot = time.time(), 0.0
        for x, y in dl:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                out = model(x)
            loss = crit(out.float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0); scaler.step(opt); scaler.update(); sched.step()
            with torch.no_grad():
                msd = model.state_dict()
                for k, v in ema.state_dict().items():
                    v.mul_(args.ema).add_(msd[k].detach(), alpha=1 - args.ema) if v.dtype.is_floating_point else v.copy_(msd[k])
            tot += loss.item()
        msg = f"ep {ep:02d} loss {tot / len(dl):.4f} {time.time() - t0:.0f}s"
        if ep % args.val_every == 0 or ep == args.epochs:
            st = evaluate(ema, va, cache, ins, args, dev)
            msg += f" | val {st}"; log.append({"epoch": ep, "pq": st.pq, "sq": st.sq, "dq": st.dq})
            if st.pq > best:
                best = st.pq; torch.save({"state_dict": ema.state_dict(), "args": vars(args), "epoch": ep, "pq": best}, ckpt); msg += " *"
        print(msg, flush=True)
    json.dump({"best_pq": best, "log": log, "args": vars(args)}, open(os.path.join(args.work, tag + "_log.json"), "w"), indent=1)
    print(f"done fold {args.fold}: best val PQ {best:.4f}", flush=True)


def evaluate(model, stems, cache, ins, args, dev, tta=2):
    from solarfil.metric import PQStats, photo_stats
    from solarfil.annotations import Photo, photo_gt_rles
    from .decode import decode, DecCfg, to_native_rle
    tot = PQStats()
    for s in stems:
        x = torch.from_numpy(np.load(os.path.join(cache, f"{s}_x.npy")).transpose(2, 0, 1).astype(np.float32) / 255.0)[None].to(dev)
        p = predict([model], x, tta=tta)[0].cpu().numpy()
        rl = to_native_rle(decode(p[0], p[2], p[1], p[3:5], DecCfg()), p.shape[-1])
        ph = Photo(stem=s, file_name=s); ph.annotators = {k: [i["poly"] for i in v] for k, v in ins[s].items()}
        tot.add(photo_stats(photo_gt_rles(ph), rl))
    return tot


if __name__ == "__main__":
    main()

In [ ]:
import glob,os,sys,subprocess
DATA=os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json',recursive=True)[0]))
r=subprocess.run(f"{sys.executable} -m solarfil5.train5 --data_root {DATA} --work /kaggle/working --fold 0 --epochs 36 --size 1536 --crop 768 --bs 6 --workers 4 --cache_root /tmp --val_every 4",shell=True)
assert r.returncode==0